# S10 — S2 backbone + S8 cumulative TEA mechanism

Per-tweet analysis (`analysis/DEEP_NIGHT_FINDINGS.md`) showed:

- S8 wins on long F1 mean (not significantly above S7/S3) but is best on seed stability and calibration stability.
- S8's known weakness: destroys ~4 high-OOV high-SEM tweets per split. These are tweets where vocabulary adaptation (S2-style continued pretraining) helps and S8's recency anchoring hurts.
- S2 specifically beats S8 on the (hiO, hiS) cell on long. When the two disagree, S2 wins there.

S10 combines the two mechanisms at training time:
1. Start each S8 expert from the **continued-pretrained MLM backbone** (S2's backbone, available on HuggingFace at `tamarasuarezrod/longeval-s2-mlm-backbone`).
2. Train the three cumulative experts (2014, 2015, 2016) with recency weights, as in S8.
3. Average parameters with the same recency weights.

This bakes the vocabulary adaptation into the S8 model at training time, removing the need for a router at inference.

Expected effect:
- Long F1 ≥ S8 (recency + vocab mechanisms both active).
- High-OOV high-SEM corner should improve (the S8 weakness).
- Seed stability should remain similar to S8 (same averaging structure).

Three possible outcomes:
1. S10 > S8 on long: proposed method.
2. S10 ≈ S8 on long: mechanisms do not compose; confirms S8 is near a local optimum.
3. S10 < S8 on long: continued pretraining destabilises S8's averaging; useful negative result.

In [1]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))

In [2]:
import copy
import json
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import f1_score
from huggingface_hub import snapshot_download
import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback, set_seed,
)
from datasets import Dataset

DATA_DIR = Path(snapshot_download(repo_id='tamarasuarezrod/longeval-data', repo_type='dataset'))

# THE ONLY CHANGE FROM S8: use the continued-pretrained MLM backbone
MODEL_NAME = 'tamarasuarezrod/longeval-s2-mlm-backbone'
SEEDS      = [42, 1, 2]
STRATEGY   = 's10-s2backbone-s8mechanism'
RECENCY_W  = {2014: 0.5, 2015: 0.75, 2016: 1.0}
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, '  Backbone:', MODEL_NAME)

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Device: cuda   Backbone: tamarasuarezrod/longeval-s2-mlm-backbone


In [3]:
def load_split(path, label_col='label'):
    with open(path) as f:
        records = json.load(f)
    df = pd.DataFrame(records).rename(columns={label_col: 'label'})
    return df[['pp_text', 'label']]

with open(DATA_DIR / 'train_eval/train.json') as f:
    train_full = pd.DataFrame(json.load(f)).rename(columns={'distant_label': 'label'})
train_full['year'] = train_full['created_at'].str.extract(r'(20\d\d)').astype(int)
print(train_full['year'].value_counts().sort_index())

eval_df = load_split(DATA_DIR / 'train_eval/interim_eval_2016.json', label_col='distant_label')
test_dfs = {
    'test_within': load_split(DATA_DIR / 'test/interim_test_2016.json'),
    'test_short':  load_split(DATA_DIR / 'test/interim_test_2018.json'),
    'test_long':   load_split(DATA_DIR / 'test/interim_test_2021.json'),
}

label2id = {l: i for i, l in enumerate(sorted(train_full['label'].unique()))}
id2label = {v: k for k, v in label2id.items()}
num_labels = len(label2id)
print('Labels:', label2id)

year
2014    16523
2015    15013
2016    18065
2017        7
Name: count, dtype: int64
Labels: {'negative': 0, 'positive': 1}


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_dataset(df):
    d = Dataset.from_dict({'text': df['pp_text'].tolist(),
                           'label': df['label'].map(label2id).tolist()})
    return d.map(lambda x: tokenizer(x['text'], truncation=True, padding='max_length', max_length=128),
                 batched=True, remove_columns=['text'])

def make_weighted_dataset(df):
    d = Dataset.from_dict({'text': df['pp_text'].tolist(),
                           'label': df['label'].map(label2id).tolist(),
                           'sample_weight': df['sample_weight'].tolist()})
    return d.map(lambda x: tokenizer(x['text'], truncation=True, padding='max_length', max_length=128),
                 batched=True, remove_columns=['text'])

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        w = inputs.pop('sample_weight', None)
        outputs = model(**inputs)
        loss_fct = torch.nn.CrossEntropyLoss(reduction='none')
        loss = loss_fct(outputs.logits, labels)
        if w is not None:
            loss = (loss * w.to(loss.device)).mean()
        else:
            loss = loss.mean()
        return (loss, outputs) if return_outputs else loss

eval_ds  = make_dataset(eval_df)
test_ds  = {n: make_dataset(df) for n, df in test_dfs.items()}

config.json:   0%|          | 0.00/676 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/359 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1344 [00:00<?, ? examples/s]

Map:   0%|          | 0/908 [00:00<?, ? examples/s]

Map:   0%|          | 0/908 [00:00<?, ? examples/s]

Map:   0%|          | 0/908 [00:00<?, ? examples/s]

In [5]:
all_results  = {}
saved_models = {}

for SEED in SEEDS:
    print(f'\n{"="*55}  SEED {SEED}')

    def train_expert(cutoff_year):
        sub = train_full[train_full['year'] <= cutoff_year].copy()
        sub['sample_weight'] = sub['year'].map(RECENCY_W).fillna(1.0)
        ds = make_weighted_dataset(sub)
        set_seed(SEED)
        m = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id,
            ignore_mismatched_sizes=True)  # MLM backbone has no classifier head
        args = TrainingArguments(
            output_dir=f'/tmp/{STRATEGY}_seed{SEED}/{cutoff_year}',
            num_train_epochs=25,
            per_device_train_batch_size=32, per_device_eval_batch_size=64,
            learning_rate=2e-05, warmup_ratio=0.1, weight_decay=0.01,
            eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
            load_best_model_at_end=True, metric_for_best_model='eval_loss', greater_is_better=False,
            logging_steps=50, fp16=(DEVICE == 'cuda'), seed=SEED, report_to='none',
            remove_unused_columns=False)
        tr = WeightedTrainer(model=m, args=args,
                             train_dataset=ds, eval_dataset=eval_ds,
                             processing_class=tokenizer,
                             callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
        tr.train()
        print(f'  Expert {cutoff_year}: trained on {len(sub):,} examples')
        return m

    experts = {yr: train_expert(yr) for yr in [2014, 2015, 2016]}

    total = sum(RECENCY_W.values())
    avg_sd = copy.deepcopy(experts[2016].state_dict())
    for k in avg_sd:
        avg_sd[k] = sum(RECENCY_W[yr] * experts[yr].state_dict()[k].float()
                        for yr in RECENCY_W) / total
    avg_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id,
        ignore_mismatched_sizes=True)
    avg_model.load_state_dict(avg_sd)
    avg_model.to(DEVICE)

    eval_trainer = Trainer(
        model=avg_model,
        args=TrainingArguments(output_dir=f'/tmp/{STRATEGY}_eval',
                               per_device_eval_batch_size=64,
                               fp16=(DEVICE == 'cuda'), report_to='none'),
        processing_class=tokenizer)

    seed_results = {}
    for name, ds in test_ds.items():
        preds_out = eval_trainer.predict(ds)
        f1 = f1_score(np.array(ds['label']), np.argmax(preds_out.predictions, axis=1), average='macro')
        seed_results[name] = f1
        print(f'  {name}: F1={f1:.4f}')

    all_results[SEED]  = seed_results
    saved_models[SEED] = avg_model


=======================================================  SEED 42


Map:   0%|          | 0/16523 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: tamarasuarezrod/longeval-s2-mlm-backbone
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

  test_within: F1=0.7350


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


  test_short: F1=0.6817


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


  test_long: F1=0.7053


In [6]:
splits = ['test_within', 'test_short', 'test_long']
print(f"{'seed':>8}  {'within':>7}  {'short':>7}  {'long':>7}  {'RPD_short':>10}  {'RPD_long':>9}")
for seed, r in all_results.items():
    rpd_s = (r['test_short']  - r['test_within']) / r['test_within']
    rpd_l = (r['test_long']   - r['test_within']) / r['test_within']
    print(f"{seed:>8}  {r['test_within']:>7.4f}  {r['test_short']:>7.4f}  {r['test_long']:>7.4f}  {rpd_s:>+10.4f}  {rpd_l:>+9.4f}")
print()
for sp in splits:
    vals = [all_results[s][sp] for s in all_results]
    print(f"{sp}: mean={np.mean(vals):.4f}  std={np.std(vals,ddof=1):.4f}  [{min(vals):.4f}–{max(vals):.4f}]")

    seed   within    short     long   RPD_short   RPD_long
      42   0.7339   0.6850   0.7109     -0.0667    -0.0313
       1   0.7288   0.6838   0.7197     -0.0617    -0.0124
       2   0.7350   0.6817   0.7053     -0.0725    -0.0403

test_within: mean=0.7326  std=0.0033  [0.7288–0.7350]
test_short: mean=0.6835  std=0.0017  [0.6817–0.6850]
test_long: mean=0.7120  std=0.0073  [0.7053–0.7197]


In [7]:
for seed, model in saved_models.items():
    repo = f'tamarasuarezrod/longeval-{STRATEGY}-seed{seed}'
    model.push_to_hub(repo)
    tokenizer.push_to_hub(repo)
    print('Saved:', repo)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Saved: tamarasuarezrod/longeval-s10-s2backbone-s8mechanism-seed42


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Saved: tamarasuarezrod/longeval-s10-s2backbone-s8mechanism-seed2
